In [ ]:
# ---------------------------------------------------------------------------
# Path setup — fixes imports and data paths when notebook runs from subdir
# ---------------------------------------------------------------------------
from pathlib import Path
import sys, os

REPO_ROOT = Path().resolve().parent.parent  # notebooks/analysis/ -> root
os.chdir(REPO_ROOT)                          # all relative paths (cache/, data/, results/, splits/) resolve from root
sys.path.insert(0, str(REPO_ROOT / "src"))   # find rise, crise, synth_gen modules

# Demographic Saliency Analysis

**Research question:** Does ArcFace rely on different facial regions for different demographic groups?

Uses the 1,680+ CRISE saliency maps already computed — zero new saliency computation required.

**Method:**
1. Run InsightFace's built-in gender/age estimator on each gallery image
2. Split existing CRISE maps by estimated gender (and age bracket)
3. Compute mean per-region importance (5-zone framework) per group
4. Compare profiles with bar charts and permutation significance tests

**Why this matters:** Standard FR bias research reports *accuracy gaps* (error rates differ by group). CRISE-ID goes deeper — it asks *why* accuracy differs by revealing whether the model uses structurally different facial evidence for different groups. Over-reliance on skin texture (forehead/jaw) vs. geometric landmarks (eye zone/nose) is a mechanistic bias claim, not just a statistical one.

**Output:** `results/demographic_analysis/`

In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

SPLIT_PATH    = "splits/lfw_1N_split.json"
GALLERY_EMB   = "cache/G.npy"
GALLERY_IDS   = "cache/gallery_ids.npy"
CRISE_SUMMARY = "results/crise_maps/summary_crise_tau0.1_N1000_s8_p0.5_MASTERSEED123.csv"
OUT_DIR       = "results/demographic_analysis"

# Age brackets (years)
AGE_BRACKETS  = [(0, 35, "<35"), (35, 55, "35-55"), (55, 120, "55+")]

# Permutation test
N_PERMUTATIONS = 10_000
PERM_SEED      = 42

import os
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------

import json
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm
from insightface.app import FaceAnalysis

print("Imports OK")

In [ ]:
# ---------------------------------------------------------------------------
# InsightFace setup
# buffalo_l includes: detection, 5-point landmarks, recognition, AND
# attribute estimation (gender + age) via the 'genderage' model.
# ---------------------------------------------------------------------------

app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)
app.prepare(ctx_id=0, det_size=(640, 640))
print("Models loaded:", list(app.models.keys()))
assert "genderage" in app.models, "genderage model not found — check buffalo_l installation"

In [ ]:
# ---------------------------------------------------------------------------
# Run gender/age estimation on all gallery images
# Cached to OUT_DIR/gallery_attributes.csv — skips if already exists.
# ---------------------------------------------------------------------------

ATTR_CACHE = os.path.join(OUT_DIR, "gallery_attributes.csv")

with open(SPLIT_PATH) as f:
    split = json.load(f)
gallery = split["gallery"]   # {identity: image_path}

gallery_ids = np.load(GALLERY_IDS, allow_pickle=True).tolist()

if os.path.exists(ATTR_CACHE):
    attr_df = pd.read_csv(ATTR_CACHE)
    print(f"Loaded cached attributes: {len(attr_df)} identities")
else:
    records = []
    for identity, img_path in tqdm(gallery.items(), desc="Gender/age estimation"):
        img = cv2.imread(img_path)
        if img is None:
            records.append({"identity": identity, "gender": None,
                            "age": None, "det_ok": False})
            continue
        faces = app.get(img)
        if not faces:
            records.append({"identity": identity, "gender": None,
                            "age": None, "det_ok": False})
            continue
        # Take the largest detected face
        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
        records.append({
            "identity": identity,
            "gender":   int(face.gender),   # 0 = female, 1 = male
            "age":      int(face.age),
            "det_ok":   True,
        })

    attr_df = pd.DataFrame(records)
    attr_df.to_csv(ATTR_CACHE, index=False)
    print(f"Saved: {ATTR_CACHE}")

# Gender label map: InsightFace outputs 0=female, 1=male
attr_df["gender_label"] = attr_df["gender"].map({0: "Female", 1: "Male"})

# Age bracket
def age_bracket(age):
    if pd.isna(age): return None
    for lo, hi, label in AGE_BRACKETS:
        if lo <= age < hi:
            return label
    return None

attr_df["age_bracket"] = attr_df["age"].apply(age_bracket)

print("\nGender distribution:")
print(attr_df["gender_label"].value_counts().to_string())
print("\nAge bracket distribution:")
print(attr_df["age_bracket"].value_counts().sort_index().to_string())
print(f"\nDetection failures: {(~attr_df['det_ok']).sum()}")

In [ ]:
# ---------------------------------------------------------------------------
# 5-zone region masks (112×112 ArcFace-aligned chip)
# Identical to demo_personal.ipynb Panel B
# ---------------------------------------------------------------------------

def make_region_masks_112():
    masks = {}
    m = np.zeros((112, 112), dtype=bool)
    m[:] = False; m[5:38,  12:100] = True; masks["Forehead"]  = m.copy()
    m[:] = False; m[35:62, 12:100] = True; masks["Eye zone"]  = m.copy()
    m[:] = False; m[58:82, 28:84]  = True; masks["Nose"]      = m.copy()
    m[:] = False; m[78:100,22:90]  = True; masks["Mouth"]     = m.copy()
    m[:] = False; m[97:112,10:102] = True; masks["Jaw/chin"]  = m.copy()
    return masks

REGION_MASKS  = make_region_masks_112()
REGION_NAMES  = list(REGION_MASKS.keys())
REGION_COLORS = ["#4E79A7", "#F28E2B", "#E15759", "#76B7B2", "#59A14F"]

def region_importance(sal):
    total = sal.sum() + 1e-9
    return {name: float(sal[mask].sum() / total)
            for name, mask in REGION_MASKS.items()}

print("Region masks defined:", REGION_NAMES)

In [ ]:
# ---------------------------------------------------------------------------
# Load CRISE summary + compute per-identity mean saliency and region importance
# One map per identity (mean over probes if multiple exist)
# ---------------------------------------------------------------------------

crise_df = pd.read_csv(CRISE_SUMMARY)
crise_df = crise_df[crise_df["saliency_path"].notna()].copy()
crise_df = crise_df[crise_df["saliency_path"].apply(
    lambda p: isinstance(p, str) and os.path.exists(p)
)].copy()

print(f"CRISE maps available: {len(crise_df)}")

# Per-identity: average saliency maps across probes, then compute region importance
id_rows = []

for identity, grp in tqdm(crise_df.groupby("true_id"), desc="Computing region importance"):
    sal_maps = []
    for sal_path in grp["saliency_path"]:
        try:
            sal_maps.append(np.load(sal_path).astype(np.float32))
        except Exception:
            pass
    if not sal_maps:
        continue
    mean_sal = np.mean(sal_maps, axis=0)
    ri = region_importance(mean_sal)
    ri["identity"]   = identity
    ri["n_probes"]   = len(sal_maps)
    ri["w_clean_mean"] = float(grp["w_clean"].mean())
    id_rows.append(ri)

ri_df = pd.DataFrame(id_rows)
print(f"Identities with region importance: {len(ri_df)}")
print(ri_df[REGION_NAMES].describe().round(3).to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Merge region importance with demographic attributes
# ---------------------------------------------------------------------------

merged = ri_df.merge(attr_df[["identity", "gender_label", "age_bracket", "age", "det_ok"]],
                     on="identity", how="inner")
merged = merged[merged["det_ok"] & merged["gender_label"].notna()].copy()

print(f"Merged: {len(merged)} identities with demographics")
print("\nGroup sizes:")
print(merged["gender_label"].value_counts().to_string())

# Save full merged table
merged.to_csv(os.path.join(OUT_DIR, "identity_region_demographics.csv"), index=False)
print(f"\nSaved: identity_region_demographics.csv")

In [ ]:
# ---------------------------------------------------------------------------
# Gender analysis — mean per-region importance by group
# ---------------------------------------------------------------------------

gender_groups = {g: merged[merged["gender_label"] == g] for g in ["Female", "Male"]}
gender_means  = {g: df[REGION_NAMES].mean() for g, df in gender_groups.items()}
gender_stds   = {g: df[REGION_NAMES].std()  for g, df in gender_groups.items()}

print("Mean per-region importance by gender:")
summary_df = pd.DataFrame(gender_means).T
print(summary_df.round(3).to_string())
print("\nDifference (Male - Female):")
diff = gender_means["Male"] - gender_means["Female"]
print(diff.round(3).to_string())

# Identify most divergent region
max_diff_region = diff.abs().idxmax()
print(f"\nMost divergent region: {max_diff_region} (Δ={diff[max_diff_region]:+.3f})")

In [ ]:
# ---------------------------------------------------------------------------
# Permutation significance test — per region, per gender comparison
# H0: group label is uninformative (permute labels, recompute mean difference)
# p-value = fraction of permutations with |diff| >= observed |diff|
# ---------------------------------------------------------------------------

rng_perm = np.random.default_rng(PERM_SEED)
labels   = merged["gender_label"].values
n_f      = (labels == "Female").sum()

perm_rows = []
obs_diffs = {}

for region in REGION_NAMES:
    vals     = merged[region].values
    obs_diff = float(vals[labels == "Male"].mean() - vals[labels == "Female"].mean())
    obs_diffs[region] = obs_diff

    null_diffs = np.empty(N_PERMUTATIONS)
    for i in range(N_PERMUTATIONS):
        perm   = rng_perm.permutation(len(vals))
        perm_f = vals[perm[:n_f]].mean()
        perm_m = vals[perm[n_f:]].mean()
        null_diffs[i] = perm_m - perm_f

    p_val = float((np.abs(null_diffs) >= abs(obs_diff)).mean())
    perm_rows.append({"Region": region, "Male mean": gender_means["Male"][region],
                      "Female mean": gender_means["Female"][region],
                      "Δ (M-F)": obs_diff, "p-value": p_val,
                      "Significant": "*" if p_val < 0.05 else ""})

perm_df = pd.DataFrame(perm_rows).set_index("Region")
print(f"Permutation test results (n_permutations={N_PERMUTATIONS}):")
print(perm_df.round(4).to_string())
perm_df.to_csv(os.path.join(OUT_DIR, "gender_permutation_test.csv"))
print(f"\nSaved: gender_permutation_test.csv")

In [ ]:
# ---------------------------------------------------------------------------
# Figure 1 — Gender: side-by-side bar chart with error bars
# ---------------------------------------------------------------------------

x     = np.arange(len(REGION_NAMES))
width = 0.35
GENDER_COLORS = {"Female": "#E15759", "Male": "#4E79A7"}

fig, ax = plt.subplots(figsize=(9, 4.5))

for i, (gender, color) in enumerate(GENDER_COLORS.items()):
    means = [gender_means[gender][r] for r in REGION_NAMES]
    stds  = [gender_stds[gender][r]  for r in REGION_NAMES]
    n     = len(gender_groups[gender])
    offset = (i - 0.5) * width
    bars = ax.bar(x + offset, means, width, label=f"{gender} (n={n})",
                  color=color, alpha=0.85, yerr=stds, capsize=3, ecolor="gray")

# Mark significant regions
for j, region in enumerate(REGION_NAMES):
    row = perm_df.loc[region]
    if row["Significant"] == "*":
        ymax = max(gender_means["Female"][region], gender_means["Male"][region])
        ax.text(j, ymax + 0.015, "*", ha="center", va="bottom", fontsize=14,
                color="black", fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(REGION_NAMES, fontsize=11)
ax.set_ylabel("Fractional saliency mass", fontsize=10)
ax.set_title("CRISE-ID: Per-region importance by gender  (* = p < 0.05)", fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(0, None)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig1_path = os.path.join(OUT_DIR, "fig_gender_region_importance.png")
plt.savefig(fig1_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig1_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Age bracket analysis
# ---------------------------------------------------------------------------

age_groups = {}
for lo, hi, label in AGE_BRACKETS:
    grp = merged[merged["age_bracket"] == label]
    if len(grp) >= 10:   # only include brackets with enough data
        age_groups[label] = grp

print("Age bracket group sizes:")
for lbl, grp in age_groups.items():
    print(f"  {lbl}: {len(grp)}")

age_means = {lbl: grp[REGION_NAMES].mean() for lbl, grp in age_groups.items()}
age_stds  = {lbl: grp[REGION_NAMES].std()  for lbl, grp in age_groups.items()}

print("\nMean per-region importance by age bracket:")
age_summary = pd.DataFrame(age_means).T
print(age_summary.round(3).to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Figure 2 — Age: grouped bar chart
# ---------------------------------------------------------------------------

if len(age_groups) < 2:
    print("Fewer than 2 age brackets with sufficient data — skipping age figure.")
else:
    AGE_COLORS = ["#59A14F", "#F28E2B", "#B07AA1"]
    x     = np.arange(len(REGION_NAMES))
    width = 0.7 / len(age_groups)

    fig, ax = plt.subplots(figsize=(10, 4.5))

    for i, (lbl, color) in enumerate(zip(age_groups.keys(), AGE_COLORS)):
        means  = [age_means[lbl][r] for r in REGION_NAMES]
        stds   = [age_stds[lbl][r]  for r in REGION_NAMES]
        n      = len(age_groups[lbl])
        offset = (i - len(age_groups) / 2 + 0.5) * width
        ax.bar(x + offset, means, width, label=f"{lbl} (n={n})",
               color=color, alpha=0.85, yerr=stds, capsize=3, ecolor="gray")

    ax.set_xticks(x)
    ax.set_xticklabels(REGION_NAMES, fontsize=11)
    ax.set_ylabel("Fractional saliency mass", fontsize=10)
    ax.set_title("CRISE-ID: Per-region importance by age bracket", fontsize=12)
    ax.legend(fontsize=10)
    ax.set_ylim(0, None)
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    fig2_path = os.path.join(OUT_DIR, "fig_age_region_importance.png")
    plt.savefig(fig2_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fig2_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Figure 3 — Combined: gender × region heatmap
# Rows = gender groups, Columns = regions, values = mean fractional importance
# ---------------------------------------------------------------------------

heat_data = pd.DataFrame({
    g: [gender_means[g][r] for r in REGION_NAMES]
    for g in ["Female", "Male"]
}, index=REGION_NAMES)

fig, ax = plt.subplots(figsize=(7, 3))
im = ax.imshow(heat_data.T.values, cmap="YlOrRd", aspect="auto", vmin=0)
plt.colorbar(im, ax=ax, label="Mean fractional saliency mass")

ax.set_xticks(range(len(REGION_NAMES)))
ax.set_xticklabels(REGION_NAMES, fontsize=10)
ax.set_yticks(range(len(["Female", "Male"])))
ax.set_yticklabels(["Female", "Male"], fontsize=10)

for i, g in enumerate(["Female", "Male"]):
    for j, r in enumerate(REGION_NAMES):
        val = gender_means[g][r]
        sig = "*" if perm_df.loc[r, "Significant"] == "*" else ""
        ax.text(j, i, f"{val:.3f}{sig}", ha="center", va="center", fontsize=9)

ax.set_title("Per-region saliency by gender  (* = p < 0.05, permutation test)", fontsize=11)
plt.tight_layout()
fig3_path = os.path.join(OUT_DIR, "fig_gender_heatmap.png")
plt.savefig(fig3_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig3_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Figure 4 — ArcFace similarity by group
# Checks whether demographic differences in saliency correlate with
# differences in recognition confidence (w_clean = cosine sim to gallery)
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Gender
for gender, color in GENDER_COLORS.items():
    vals = gender_groups[gender]["w_clean_mean"].dropna()
    axes[0].hist(vals, bins=30, alpha=0.6, label=f"{gender} (n={len(vals)})",
                 color=color, density=True)
axes[0].set_xlabel("Mean ArcFace similarity (w_clean)", fontsize=10)
axes[0].set_ylabel("Density", fontsize=10)
axes[0].set_title("Recognition confidence by gender", fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Age
for (lo, hi, lbl), color in zip(AGE_BRACKETS, ["#59A14F", "#F28E2B", "#B07AA1"]):
    if lbl in age_groups:
        vals = age_groups[lbl]["w_clean_mean"].dropna()
        axes[1].hist(vals, bins=30, alpha=0.6, label=f"{lbl} (n={len(vals)})",
                     color=color, density=True)
axes[1].set_xlabel("Mean ArcFace similarity (w_clean)", fontsize=10)
axes[1].set_ylabel("Density", fontsize=10)
axes[1].set_title("Recognition confidence by age bracket", fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle("ArcFace recognition confidence by demographic group", fontsize=12)
plt.tight_layout()
fig4_path = os.path.join(OUT_DIR, "fig_confidence_by_group.png")
plt.savefig(fig4_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig4_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------

print("=" * 60)
print(" DEMOGRAPHIC SALIENCY ANALYSIS — SUMMARY")
print("=" * 60)
print(f"  Identities analyzed : {len(merged)}")
print(f"  Female              : {len(gender_groups['Female'])}")
print(f"  Male                : {len(gender_groups['Male'])}")
print()
print("PER-REGION IMPORTANCE (mean fractional saliency mass)")
print(f"  {'Region':<12}  {'Female':>8}  {'Male':>8}  {'Δ(M-F)':>8}  {'p':>8}")
for region in REGION_NAMES:
    f_val = gender_means["Female"][region]
    m_val = gender_means["Male"][region]
    delta = m_val - f_val
    pval  = perm_df.loc[region, "p-value"]
    sig   = " *" if pval < 0.05 else ""
    print(f"  {region:<12}  {f_val:>8.3f}  {m_val:>8.3f}  {delta:>+8.3f}  {pval:>8.4f}{sig}")
print()
sig_regions = [r for r in REGION_NAMES if perm_df.loc[r, "p-value"] < 0.05]
if sig_regions:
    print(f"  Significant regions (p<0.05): {', '.join(sig_regions)}")
    print()
    print("INTERPRETATION")
    print("  ArcFace relies on structurally different facial evidence")
    print("  for different demographic groups. This is not just an accuracy")
    print("  gap — it is a mechanistic difference in the decision basis,")
    print("  invisible from confidence scores alone.")
else:
    print("  No regions reached p<0.05. Region profiles are broadly similar")
    print("  across gender groups for this dataset.")
print("=" * 60)